In [0]:
spark.sql("SHOW TABLES IN uber.silver").show()

In [0]:
%sql
DROP TABLE IF EXISTS uber_project.default.staging_rides;
DROP TABLE IF EXISTS uber_project.default.silver_obt;


In [0]:
%sql
DROP TABLE IF EXISTS uber.bronze.staging_rides;
DROP TABLE IF EXISTS uber.bronze.silver_rides;

In [0]:
%sql
DROP TABLE IF EXISTS uber.bronze.staging_rides;

In [0]:
%sql
DROP TABLE IF EXISTS `uber`.`silver`.`silver_obt`;

In [0]:
%sql
select * from uber.bronze.streaming_raw_rides

In [0]:
%sql
select * from uber.bronze.bulk_rides

In [0]:
%sql
select * from uber.bronze.staging_rides

In [0]:
%sql
select * from uber.bronze.map_cities

In [0]:
%sql
select * from uber.bronze.silver_obt    


In [0]:
%sql
-- Please edit the sample below

CREATE OR REFRESH STREAMING TABLE
silver_obt
AS 

SELECT
    
        staging_rides.ride_id, staging_rides.confirmation_number, staging_rides.passenger_id, staging_rides.driver_id, staging_rides.vehicle_id, staging_rides.pickup_location_id, staging_rides.dropoff_location_id, staging_rides.vehicle_type_id, staging_rides.vehicle_make_id, staging_rides.payment_method_id, staging_rides.ride_status_id, staging_rides.pickup_city_id, staging_rides.dropoff_city_id, staging_rides.cancellation_reason_id, staging_rides.passenger_name, staging_rides.passenger_email, staging_rides.passenger_phone, staging_rides.driver_name, staging_rides.driver_rating, staging_rides.driver_phone, staging_rides.driver_license, staging_rides.vehicle_model, staging_rides.vehicle_color, staging_rides.license_plate, staging_rides.pickup_address, staging_rides.pickup_latitude, staging_rides.pickup_longitude, staging_rides.dropoff_address, staging_rides.dropoff_latitude, staging_rides.dropoff_longitude, staging_rides.distance_miles, staging_rides.duration_minutes, staging_rides.booking_timestamp, staging_rides.pickup_timestamp, staging_rides.dropoff_timestamp, staging_rides.base_fare, staging_rides.distance_fare, staging_rides.time_fare, staging_rides.surge_multiplier, staging_rides.subtotal, staging_rides.tip_amount, staging_rides.total_fare, staging_rides.rating
            
                ,
            
    
        map_vehicle_makes.vehicle_make
            
                ,
            
    
        map_vehicle_types.vehicle_type,map_vehicle_types.description,map_vehicle_types.base_rate,map_vehicle_types.per_mile,map_vehicle_types.per_minute
            
                ,
            
    
        map_ride_statuses.ride_status
            
                ,
            
    
        map_payment_methods.payment_method, map_payment_methods.is_card, map_payment_methods.requires_auth
            
                ,
            
    
        map_cities.city as pickup_city, map_cities.state, map_cities.region, map_cities.updated_at as city_updated_at
            
                ,
            
    
        map_cancellation_reasons.cancellation_reason
            
    
FROM 
    STREAM (silver_staging_rides) 
    WATERMARK booking_timestamp DELAY OF INTERVAL 2 MINUTES staging_rides
        
    
        
            LEFT JOIN uber.bronze.map_vehicle_makes map_vehicle_makes ON staging_rides.vehicle_make_id = map_vehicle_makes.vehicle_make_id
        
    
        
            LEFT JOIN uber.bronze.map_vehicle_types map_vehicle_types ON staging_rides.vehicle_type_id = map_vehicle_types.vehicle_type_id
        
    
        
            LEFT JOIN uber.bronze.map_ride_statuses map_ride_statuses ON staging_rides.ride_status_id = map_ride_statuses.ride_status_id
        
    
        
            LEFT JOIN uber.bronze.map_payment_methods map_payment_methods ON staging_rides.payment_method_id = map_payment_methods.payment_method_id
        
    
        
            LEFT JOIN uber.bronze.map_cities map_cities ON staging_rides.pickup_city_id = map_cities.city_id
        
    
        
            LEFT JOIN uber.bronze.map_cancellation_reasons map_cancellation_reasons ON staging_rides.cancellation_reason_id = map_cancellation_reasons.cancellation_reason_id


In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import *
from pyspark.sql.types import *

# schema for streaming table raw_rides
rides_schema = StructType([StructField('ride_id', StringType(), True), StructField('confirmation_number', StringType(), True), StructField('passenger_id', StringType(), True), StructField('driver_id', StringType(), True), StructField('vehicle_id', StringType(), True), StructField('pickup_location_id', StringType(), True), StructField('dropoff_location_id', StringType(), True), StructField('vehicle_type_id', LongType(), True), StructField('vehicle_make_id', LongType(), True), StructField('payment_method_id', LongType(), True), StructField('ride_status_id', LongType(), True), StructField('pickup_city_id', LongType(), True), StructField('dropoff_city_id', LongType(), True), StructField('cancellation_reason_id', LongType(), True), StructField('passenger_name', StringType(), True), StructField('passenger_email', StringType(), True), StructField('passenger_phone', StringType(), True), StructField('driver_name', StringType(), True), StructField('driver_rating', DoubleType(), True), StructField('driver_phone', StringType(), True), StructField('driver_license', StringType(), True), StructField('vehicle_model', StringType(), True), StructField('vehicle_color', StringType(), True), StructField('license_plate', StringType(), True), StructField('pickup_address', StringType(), True), StructField('pickup_latitude', DoubleType(), True), StructField('pickup_longitude', DoubleType(), True), StructField('dropoff_address', StringType(), True), StructField('dropoff_latitude', DoubleType(), True), StructField('dropoff_longitude', DoubleType(), True), StructField('distance_miles', DoubleType(), True), StructField('duration_minutes', LongType(), True), StructField('booking_timestamp', TimestampType(), True), StructField('pickup_timestamp', StringType(), True), StructField('dropoff_timestamp', StringType(), True), StructField('base_fare', DoubleType(), True), StructField('distance_fare', DoubleType(), True), StructField('time_fare', DoubleType(), True), StructField('surge_multiplier', DoubleType(), True), StructField('subtotal', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('total_fare', DoubleType(), True), StructField('rating', DoubleType(), True)])


# empty streaming table
dp.create_streaming_table("silver_staging_rides")

# Bulk initial load
@dp.append_flow(
    target="silver_staging_rides",
    once=True
)
def rides_bulk():
    df = spark.read.table("uber.bronze.bulk_rides")
    df = df.withColumn("booking_timestamp", col("booking_timestamp").cast("timestamp"))
   
    return df


# Streaming table
@dp.append_flow(
    target="silver_staging_rides",
    once=False
)
def rides_stream():
    df = spark.readStream.table("uber.bronze.streaming_raw_rides")
    df_parsed = df.withColumn("parsed_rides", from_json(col("rides"),rides_schema))\
    .select("parsed_rides.*")
    return df_parsed
